In [ ]:
options(encoding = "UTF-8")
Sys.setlocale("LC_ALL", "pl_PL.UTF-8")

# packages loader
use_package <- function(pkg) {
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg)
    library(pkg, character.only = TRUE)
  }
}

use_package('pls')  # pls - partial least squares


zadanie 1
PC - take principal components of the X matrix.

Problem: PC regression chooses explanatory without reference to Y.


1. Principal component analysis

Otrzymujemy, wartości, własne i ich wartości. Ciekawe też jest zobaczenie ile wyjaśniają całości położenia danych

In [ ]:
data(yarn)
pcxyarn = prcomp(yarn$NIR,center=TRUE,scale=TRUE)
ev = pcxyarn$sdev^2
ev_interpretable = pcxyarn$sdev^2 / sum(pcxyarn$sdev^2)

sprintf("%.2f", ev_interpretable)


2. Principal Component Regression

Robimy tutaj PCR, w ten sposób, że:
1. porównujemy jak dużo położenia danych jest wyjaśnionych przez transformacje PCR
2. porównujemy wyniki predykcji (to jes R^2???) 

In [ ]:
pcreg = pcr(density~NIR,ncomp=6,data=yarn)
summary(pcreg)
#Y = Xbeta + epsilon 
# percentage explained by model X\beta
data=yarn

3. PLS regression

Zauważmy istotnie różnice, w $R^2$. Mamy tutaj bardzo dobre dopasowanie, względem zmiennych objaśniających. (w przeciwieństwie do PCR)
- TODO - doczytać czemu tak się dzieje

In [ ]:
plsreg = plsr(density~NIR, ncomp=6, data = yarn)
summary(plsreg)

4. 

Teraz stosujemy metody bez wycentrowania

Tutaj dostajemy zupełnie inne wyniki, ponieważ z powodu braku standaryzacji, są zmienne które są bardzo rozciągnięte, i one dominują, pomimo, że nie muszą wnosić największej informacji, np.
- wiek ($xe+1$)
- zarobki ($xe+4$)

Bez centrowania, otrzymujemy, błędne wyniki, ponieważ korzystamy z drugiego, nie wycentrowanego momentu, z tego powodu otrzymujemy większe wartości, ponieważ zachodzi
$$\sum_i (x_i - \hat{x})^2 \leq \sum_i (x_i - c)^2$$

In [ ]:
data(yarn)
pcxyarn = prcomp(yarn$NIR,center=TRUE,scale=FALSE)
ev = pcxyarn$sdev^2
ev_interpretable = pcxyarn$sdev^2 / sum(pcxyarn$sdev^2)

sprintf("%.2f", ev_interpretable)

In [ ]:
data(yarn)
pcxyarn = prcomp(yarn$NIR,center=FALSE,scale=FALSE)
ev = pcxyarn$sdev^2
ev_interpretable = pcxyarn$sdev^2 / sum(pcxyarn$sdev^2)

sprintf("%.2f", ev_interpretable)

Zadanie 2

In [ ]:
data("yarn")
data("oliveoil")
data("gasoline")

gasTrain <- gasoline[1:50, ]
gasTest <- gasoline[51:60, ]
gas1 <- plsr(octane ~ NIR, ncomp = 10, data = gasTrain, validation = "LOO")
summary(gas1)

plot(RMSEP(gas1), legendpos = "topright")
plot(gas1, ncomp = 2, asp = 1, line = TRUE)
plot(gas1, plottype = "scores", comps = 1:3)

explvar(gas1)

plot(gas1, "loadings", comps = 1:2, legendpos = "topleft",
     labels = "numbers", xlab = "nm")
abline(h = 0)

predict(gas1, ncomp = 2, newdata = gasTest)

RMSEP(gas1, newdata = gasTest)

dens1 <- plsr(density ~ NIR, ncomp = 5, data = yarn)

dim(oliveoil$sensory)
plsr(sensory ~ chemical, data = oliveoil)

trainind <- which(yarn$train == TRUE)
dens2 <- update(dens1, subset = trainind)
dens3 <- update(dens1, ncomp = 10)

olive1 <- plsr(sensory ~ chemical, scale = TRUE, data = oliveoil)

gas2 <- plsr(octane ~ msc(NIR), ncomp = 10, data = gasTrain)
predict(gas2, ncomp = 3, newdata = gasTest)

gas2.cv <- crossval(gas2, segments = 10)
plot(MSEP(gas2.cv), legendpos = "topright")
summary(gas2.cv, what = "validation")

plot(gas1, plottype = "coef", ncomp = 1:3, legendpos = "bottomleft",
     labels = "numbers", xlab = "nm")

predict(gas1, ncomp = 2:3, newdata = gasTest[1:5, ])

predict(gas1, comps = 2, newdata = gasTest[1:5, ])

drop(predict(gas1, ncomp = 2:3, newdata = gasTest[1:5, ]))

predplot(gas1, ncomp = 2, newdata = gasTest, asp = 1, line = TRUE)

X <- gasTrain$NIR
Y <- gasTrain$octane
ncomp <- 5
cvPreds <- matrix(nrow = nrow(X), ncol = ncomp)
for (i in 1:nrow(X)) {
  fit <- simpls.fit(X[-i, ], Y[-i], ncomp = ncomp, stripped = TRUE)
  cvPreds[i, ] <- (X[i, ] - fit$Xmeans) %*% drop(fit$coefficients) +
    fit$Ymeans
}
sqrt(colMeans((cvPreds - Y)^2))